<a href="https://colab.research.google.com/github/davidrpugh/data-analytics-with-polars/blob/main/ch14.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 14: Joining and Concatenating

This notebook accompanies Chapter 14 of *Python Polars: The Definitive Guide* by Jeroen Janssens and Thijs Nieuwdorp. It is adapted for **Polars 2.1** and **Google Colab**.

Real data rarely lives in one table. This chapter is about combining DataFrames, in two fundamentally different ways:

| Goal | Tools |
|---|---|
| Combine tables **by matching values** (like a database join) | `df.join()` (exact), `df.join_asof()` (nearest) |
| Combine tables **by sticking them together** (no matching) | `pl.concat()`, `df.vstack()`, `df.hstack()`, `df.extend()`, `series.append()` |

By the end you will be able to:

- choose the right join strategy (`inner`, `left`, `right`, `full`, `cross`, `semi`, `anti`)
- protect yourself from accidental row explosions with `validate=`
- join on the **nearest** value with `join_asof` (including per-group joins and tolerances)
- pick between `concat`, `vstack`, `hstack`, and `extend`, and understand the **memory trade-off** (chunks) between them

**Notes on this adaptation.**

1. Several results depend on row order. In Polars 2.1 the order of join results is **not guaranteed** unless you ask for it, so we either pass `maintain_order=` or sort.
3. Polars 2.x behaves differently from the book in a few places (for example horizontal concatenation of frames with different heights). The cells print what **your** version does; read the output and compare with the text.

Run the cells from top to bottom.

## 0. Setup

We pin Polars to the **2.1 series** so everyone sees the same behavior. If Polars was already imported in your Colab session, restart the session (Runtime > Restart session) so the pinned version is picked up.

In [ ]:
%pip install -q "polars==2.1.*"

### Imports and helpers

- `download` fetches a data file from the book's repository **only if it isn't already there**, and raises on HTTP errors.

In [ ]:
from pathlib import Path
import urllib.request
import warnings

import polars as pl
from polars.testing import assert_frame_equal

print(f"Polars {pl.__version__}")


BASE_URL = "https://github.com/davidrpugh/data-analytics-with-polars/raw/refs/heads/main/data"
DATA_DIR = Path("data")


def download(relative_path: str) -> Path:
    """Download BASE_URL/relative_path into DATA_DIR (unless it exists) and return the local path."""
    path = DATA_DIR / relative_path
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        temporary = path.with_name(path.name + ".tmp")
        urllib.request.urlretrieve(f"{BASE_URL}/{relative_path}", temporary)
        temporary.rename(path)
    return path


## 1. Joining

`df.join(other, ...)` combines two DataFrames based on the **values** in one or more key columns. Its main arguments:

| Argument | Meaning |
|---|---|
| `other` | the DataFrame to join with |
| `on` | key column(s) with the **same name** on both sides |
| `left_on`, `right_on` | key column(s) when the names differ |
| `how` | the join strategy (next section) |
| `suffix` | appended to right-hand columns whose names clash with left-hand ones (default `"_right"`) |
| `validate` | checks the join's **cardinality** (Section 3) |
| `coalesce` | whether to merge the key columns into one |
| `maintain_order` | which side's row order to preserve in the result |
| null handling | by default, null keys do **not** match each other (see below) |

### Join strategies

| `how` | Keeps |
|---|---|
| `inner` (default) | only rows with a match in **both** frames |
| `left` | all left rows; right values filled with null if there is no match |
| `right` | all right rows; left values filled with null if there is no match |
| `full` | all rows from both frames |
| `cross` | every combination of a left row and a right row (the Cartesian product) |
| `semi` | left rows that **have** a match (no right columns are added) |
| `anti` | left rows that have **no** match (no right columns are added) |

We demonstrate them with two tiny tables. Keys `B`, `C`, `D` appear in both; `A` only on the left; `E` only on the right.

In [ ]:
df_left = pl.DataFrame({"key": ["A", "B", "C", "D"], "value": [1, 2, 3, 4]})
df_right = pl.DataFrame({"key": ["B", "C", "D", "E"], "value": [5, 6, 7, 8]})

print(df_left)
print(df_right)

### Inner

Only keys present on **both** sides survive: `B`, `C`, `D`. Both tables have a column called `value`, so the right-hand one gets the default suffix `_right`.

Because the order of join results is not guaranteed, we sort by `key` to make the output reproducible.

In [ ]:
df_left.join(df_right, on="key", how="inner").sort("key")

### Full

Keep every row of both frames; fill what is missing with null. In the book's example the suffix is changed to `_other`. Notice that a full join returns **two key columns** (`key` and `key_other`); one of them is null for rows that came from only one side. Pass `coalesce=True` to merge them into a single key column, which is usually what you want.

In [ ]:
df_left.join(df_right, on="key", how="full", suffix="_other").sort("key", nulls_last=True)

In [ ]:
df_left.join(df_right, on="key", how="full", coalesce=True).sort("key")

### Left

All rows from the left frame; matching right values, or null where there is no match (here for key `A`).

> **Row order.** The book says a left join preserves the order of the left frame. That is how it behaves in practice, but the documented default for `maintain_order` is *not* to guarantee any order. If you rely on the order, **ask for it** with `maintain_order="left"`. The next cell shows what your version does when you do.

In [ ]:
left_joined = df_left.join(df_right, on="key", how="left", maintain_order="left")
left_joined

### Right

The mirror image of `left`: all rows of the right frame are kept. Note that the key column of a right join comes from the right frame, and column order changes accordingly.

In [ ]:
df_left.join(df_right, on="key", how="right").sort("key")

### Cross

A cross join pairs **every** left row with **every** right row, so the result has `len(left) * len(right)` rows. No `on` is needed. Beware: this grows quickly (two tables of 10,000 rows give 100 million rows).

In [ ]:
cross = df_left.join(df_right, how="cross")
cross

In [ ]:
assert cross.height == df_left.height * df_right.height

### Semi and anti

These two are **filtering joins**: they never add columns from the right frame; they only decide which left rows to keep.

- `semi`: keep left rows whose key **is** in the right frame
- `anti`: keep left rows whose key is **not** in the right frame

In [ ]:
df_left.join(df_right, on="key", how="semi").sort("key")

In [ ]:
df_left.join(df_right, on="key", how="anti").sort("key")

### Different key names: `left_on` and `right_on`

When the key columns have different names, use `left_on` and `right_on`. Both key columns are kept unless you `coalesce`; for an inner or left join Polars keeps only the left key name.

In [ ]:
orders = pl.DataFrame({"order_id": [1, 2, 3], "customer": ["ann", "bob", "ann"]})
customers = pl.DataFrame({"name": ["ann", "bob", "cy"], "city": ["Oslo", "Rome", "Lima"]})

orders.join(customers, left_on="customer", right_on="name", how="left", validate="m:1").sort("order_id")

### Keys must have the same data type

Joining an integer key to a string key is an error, not an implicit conversion. Cast one side first (`pl.col("id").cast(pl.Int64)`, or `str.to_integer()` for text that holds numbers).

In [ ]:
ints = pl.DataFrame({"id": [1, 2, 3], "a": ["x", "y", "z"]})
strs = pl.DataFrame({"id": ["1", "2", "4"], "b": [10, 20, 40]})

ints.join(strs, on="id", how="inner")

In [ ]:
print("After casting the string key:")
ints.join(strs.with_columns(pl.col("id").cast(pl.Int64)), on="id", how="inner").sort("id")

### Null keys

By default, **null keys never match each other** (as in SQL). If you want `null == null` to count as a match, there is an option for it. Its name changed across Polars versions (`join_nulls` in the book, `nulls_equal` in newer releases), so the cell below tries the new name and prints any warning or error.

In [ ]:
with_nulls_left = pl.DataFrame({"key": ["a", None], "x": [1, 2]})
with_nulls_right = pl.DataFrame({"key": ["a", None], "y": [10, 20]})

print("Default (nulls do not match):")
print(with_nulls_left.join(with_nulls_right, on="key", how="inner"))

In [ ]:
print("nulls_equal=True:")
equal_nulls = with_nulls_left.join(with_nulls_right, on="key", how="inner", nulls_equal=True)
print(equal_nulls)

## 2. Joining on multiple columns

Pass a **list** of column names to `on` to require a match on **all** of them. Here `name` alone is not enough: both `name` *and* `city` must agree. Dave lives in `SF` in one table and `Chicago` in the other, so he is dropped.

In [ ]:
residences_left = pl.DataFrame(
    {
        "name": ["Alice", "Bob", "Charlie", "Dave"],
        "city": ["NY", "LA", "NY", "SF"],
        "age": [25, 30, 35, 40],
    }
)

departments_right = pl.DataFrame(
    {
        "name": ["Alice", "Bob", "Charlie", "Dave"],
        "city": ["NY", "LA", "NY", "Chicago"],
        "department": ["Finance", "Marketing", "Engineering", "Operations"],
    }
)

residences_left.join(departments_right, on=["name", "city"], how="inner").sort("name")

## 3. Validation: check your assumptions

Every join makes an assumption about how rows relate: *each employee belongs to exactly one department*, *each customer ID appears once*, and so on. If the assumption is wrong (say a department appears twice), the join silently **multiplies rows** and every later sum is inflated.

`validate=` makes Polars **check the assumption** and raise an error if it fails. It is cheap insurance, and a best practice for every join.

| `validate` | Left keys | Right keys | Example |
|---|---|---|---|
| `"m:m"` (default) | may repeat | may repeat | employees and projects |
| `"1:m"` | must be unique | may repeat | one department, many employees |
| `"m:1"` | may repeat | must be unique | many employees, one department each |
| `"1:1"` | must be unique | must be unique | employees and their ID badges |

`"m:m"` is the default and checks nothing.

Here many employees share a department (`m:1` from employees to departments).

In [ ]:
employees = pl.DataFrame(
    {
        "employee_id": [1, 2, 3, 4],
        "name": ["Alice", "Bob", "Charlie", "Dave"],
        "department_id": [10, 10, 30, 10],
    }
)

departments = pl.DataFrame(
    {
        "department_id": [10, 20, 30],
        "department_name": ["Information Technology", "Finance", "Human Resources"],
    }
)

employees.join(departments, on="department_id", how="left", validate="m:1").sort("employee_id")

The same data **fails** a `1:m` or `1:1` check, because `department_id` repeats on the left (departments 10 appears three times):

In [ ]:
print('validate="1:m":')
employees.join(departments, on="department_id", how="left", validate="1:m")


In [ ]:
print('validate="1:1":')
employees.join(departments, on="department_id", how="left", validate="1:1")

### When the right side is not unique

If a department ID shows up **twice** in the departments table, `m:1` fails. Without `validate`, the join would "succeed" and quietly give you **duplicated employee rows**. The check turns a silent data bug into an immediate, readable error.

In [ ]:
departments_with_duplicate = pl.DataFrame(
    {
        "department_id": [10, 20, 10],
        "department_name": ["Information Technology", "Finance", "Human Resources"],
    }
)

print('With validate="m:1":')
employees.join(departments_with_duplicate, on="department_id", how="left", validate="m:1")


In [ ]:
print("Without validate (default m:m), the employee rows are silently duplicated:")
unvalidated = employees.join(departments_with_duplicate, on="department_id", how="left")
unvalidated

### One-to-one

An ID-badge table has exactly one row per employee, so `1:1` holds.

In [ ]:
badges = pl.DataFrame({"employee_id": [1, 2, 3, 4], "badge": ["b-01", "b-02", "b-03", "b-04"]})

employees.join(badges, on="employee_id", how="left", validate="1:1").sort("employee_id")

### A note on non-equi joins: `join_where`

All joins above require **equality** between keys. A *non-equi* join matches rows on other conditions (`<`, `>`, ranges). Polars offers `df.join_where(other, predicate, ...)`. The book called it unstable and left it out. The cell below runs a tiny example and prints any warning your version emits, so you can see its current status.

In [ ]:
small_left = pl.DataFrame({"a": [1, 5, 10]})
small_right = pl.DataFrame({"b": [3, 7]})

pairs = small_left.join_where(small_right, pl.col("a") < pl.col("b"))
pairs

## 4. Inexact joining: `join_asof`

Sometimes keys that *should* match differ slightly: one system stamps a sale when it is written to the database, another when payment is received. `df.join_asof()` matches each left row to the **nearest** value on the right instead of an equal one.

Arguments (Table 14-2):

| Argument | Meaning |
|---|---|
| `on` / `left_on`, `right_on` | the column to match on, which must be **sorted** in both frames |
| `by` / `by_left`, `by_right` | columns that must match **exactly** first; the nearest-value search happens within each group |
| `strategy` | `"backward"` (default), `"forward"`, or `"nearest"` |
| `tolerance` | the largest allowed distance; farther matches are discarded (null) |
| `coalesce` | merge the two `on` columns into one (default `True`) |
| `suffix` | suffix for clashing column names |

> **Both inputs must be sorted on the `on` column.** `join_asof` does not sort for you. If an input is not sorted you get an error, as shown next.

In [ ]:
asof_left = pl.DataFrame({"int_id": [10, 5], "value": ["b", "a"]})
asof_right = pl.DataFrame({"int_id": [4, 7, 12], "value": [1, 2, 3]})

print("Unsorted left frame:")
asof_left.join_asof(asof_right, on="int_id", tolerance=3)

Sort first (the right frame is already sorted), then join. The default strategy is `backward`: for each left row, take the **last right row whose key is equal or smaller**.

In [ ]:
asof_left = asof_left.sort("int_id")

asof_left.join_asof(asof_right, on="int_id")

### Keeping both key columns

When the key column has the same name on both sides, `join_asof` keeps only the left one. Use `coalesce=False` to also keep the right key (it becomes `int_id_right`), which lets you **see which right row each left row matched**: 5 matched 4, and 10 matched 7.

In [ ]:
asof_left.join_asof(asof_right, on="int_id", coalesce=False)

If the key columns have different names, use `left_on` and `right_on`. Both are kept in the result.

In [ ]:
asof_left.join_asof(
    asof_right.rename({"int_id": "int_id_right"}),
    left_on="int_id",
    right_on="int_id_right",
)

### The three strategies

For the data below (left keys `5` and `10`; right keys `4`, `7`, `12`):

| Strategy | Rule | Left `5` | Left `10` |
|---|---|---|---|
| `backward` | last right key **<=** the left key | 4 | 7 |
| `forward` | first right key **>=** the left key | 7 | 12 |
| `nearest` | right key with the smallest distance | 4 | 12 |

`tolerance=3` additionally requires that the matched key is **within 3** of the left key.

Note the `nearest` result for left `10`: both `7` (distance 3) and `12` (distance 2) qualify, and `12` is closer.

In [ ]:
for strategy in ["backward", "forward", "nearest"]:
    print(f"strategy={strategy!r}")
    print(
        asof_left.join_asof(
            asof_right, on="int_id", tolerance=3, strategy=strategy, coalesce=False
        )
    )

### Tolerance

`tolerance` is the **maximum distance** between the keys. Matches farther away are discarded, and the right-hand columns become null. For numbers it is a number; for temporal columns you can use a `timedelta` or a **duration string** such as `"60d"` or `"7d12h30m"` (see Chapter 13, Table 13-5).

With `tolerance=2` and the `backward` strategy, left `5` still matches right `4` (distance 1), but left `10` would have to match right `7` (distance 3), which is too far, so its right-hand columns become null:

In [ ]:
asof_left.join_asof(asof_right, on="int_id", tolerance=2, coalesce=False)

### Matching within groups: `by`

`by` first requires an **exact** match on the grouping column(s) and only then looks for the nearest key within that group. A classic use is joining trades to the most recent quote **of the same stock**. Here each trade is matched with the latest quote at or before it, for the same ticker, but only if that quote is at most 3 seconds old.

In [ ]:
quotes = pl.DataFrame(
    {
        "time": pl.Series(
            ["2024-01-01 10:00:00", "2024-01-01 10:00:02", "2024-01-01 10:00:04", "2024-01-01 10:00:09"]
        ).str.to_datetime(),
        "ticker": ["AAA", "BBB", "AAA", "BBB"],
        "bid": [100.0, 50.0, 101.0, 51.0],
    }
)

trades = pl.DataFrame(
    {
        "time": pl.Series(
            ["2024-01-01 10:00:03", "2024-01-01 10:00:05", "2024-01-01 10:00:12"]
        ).str.to_datetime(),
        "ticker": ["AAA", "BBB", "AAA"],
        "qty": [10, 20, 30],
    }
)

# Both frames are sorted by time. With `by`, sortedness must hold within each group,
# and ensuring that is YOUR job; sorting by time guarantees it here.
trades.join_asof(quotes, on="time", by="ticker", strategy="backward", tolerance="3s", coalesce=False)

The first trade (AAA at 10:00:03) matches the AAA quote at 10:00:00. The second (BBB at 10:00:05) finds the BBB quote of 10:00:02 (3 seconds old). The third (AAA at 10:00:12) has only a quote from 10:00:04, 8 seconds old, which is outside the tolerance, so its quote columns are null.

## 5. Use case: marketing campaign attribution

You want to know whether marketing campaigns raised sales. You have:

- `campaigns.csv`: campaign name, start date-time, and the product type it targeted
- `transactions.csv`: 20,000 sales with date-time, product type, and quantity

For each sale we want the **most recent campaign for the same product type that started before the sale and is still "in effect"**, which we define as within 60 days. That is exactly a `backward` as-of join, with `by="Product Type"` and `tolerance="60d"`.

We load both files lazily with `scan_csv`, so Polars can optimize the whole query.

In [ ]:
CAMPAIGNS_CSV = download("campaigns.csv")
TRANSACTIONS_CSV = download("transactions.csv")

campaigns = pl.scan_csv(CAMPAIGNS_CSV)


In [ ]:
campaigns.select(pl.col("Product Type").unique().sort()).collect()

In [ ]:
transactions = pl.scan_csv(TRANSACTIONS_CSV)


Both timestamp columns are still **strings**. `join_asof` needs real temporal columns of the **same data type**, so we parse them. The sale dates have fractional seconds (`%.f`), and we cast to microsecond precision to match the campaign dates.

In [ ]:
transactions = transactions.with_columns(
    pl.col("Sale Date")
    .str.to_datetime("%Y-%m-%d %H:%M:%S%.f")
    .cast(pl.Datetime("us")),
)
campaigns = campaigns.with_columns(
    pl.col("Campaign Date").str.to_datetime("%Y-%m-%d %H:%M:%S"),
)

print(transactions.collect_schema())
print(campaigns.collect_schema())

### The as-of join

We sort both frames by their time column, because `join_asof` requires it.

When you use `by=`, the time column must be sorted *within each group*, and Polars may not be able to check that for you (the draft notebook passed `check_sortedness=False`, which suggests a warning about it). Because we sorted explicitly right above, passing `check_sortedness=False` is safe. The cell below first runs the join **without** that argument and prints any warning, then runs it with the argument, and checks that the two give the same answer.

In [ ]:
def attribute_campaigns(check_sortedness=None):
    options = {} if check_sortedness is None else {"check_sortedness": check_sortedness}
    return (
        transactions.sort("Sale Date")
        .join_asof(
            campaigns.sort("Campaign Date"),
            left_on="Sale Date",
            right_on="Campaign Date",
            by="Product Type",
            strategy="backward",
            tolerance="60d",
            **options,
        )
        .collect()
    )


In [ ]:
print("Without check_sortedness:")
sales_with_campaign_default = attribute_campaigns()
sales_with_campaign_default

In [ ]:
print("With check_sortedness=False:")
sales_with_campaign_df = attribute_campaigns(check_sortedness=False)
sales_with_campaign_df

In [ ]:
assert_frame_equal(sales_with_campaign_default, sales_with_campaign_df)


### Did the campaigns work?

Group by product type and campaign (Chapter 13) and compare the **average quantity** per sale. Rows where `Campaign Name` is null are sales made with no campaign in effect.

In [ ]:
(
    sales_with_campaign_df.group_by("Product Type", "Campaign Name")
    .agg(pl.col("Quantity").mean())
    .sort("Product Type", "Campaign Name")
)

Sales during a campaign average roughly 8 units, against about 5.5 without one: campaigns appear to help. The exceptions are **Toys** (no campaign was ever run) and **Books**. Why does Books have no campaign row?

Let's look at its campaigns:

In [ ]:
campaigns.filter(pl.col("Product Type") == "Books").collect()

Books had a single campaign, a clearance sale starting on **31 December 2023 at 21:00**. Are there any book sales after that moment?

In [ ]:
(
    transactions.filter(
        (pl.col("Product Type") == "Books")
        & (pl.col("Sale Date") > pl.datetime(2023, 12, 31, 21, 0, 0, time_unit="us"))
    ).collect()
)

There are none. With the `backward` strategy a sale can only match a campaign that started **before** it, so no sale was ever attributed to the clearance. The sales it might have caused are simply not in the data, which makes the campaign *look* ineffective. A reminder that a join can only reflect what the data contains: always ask *why* a group is empty before drawing conclusions.

## 6. Vertical and horizontal concatenation

`df.join()` combines frames **by value**. Sometimes you just want to add frames together. There are three ways to do that, with a different memory behavior:

1. **Copy** the data into a **new** DataFrame: `pl.concat()` (the default; it also *rechunks*).
2. **Point** the new DataFrame at the existing data: `df.vstack()`, `df.hstack()`, `series.append()`. No copy; very fast, but the data lives in several separate **chunks**.
3. **Copy the second frame behind the first**: `df.extend()`, which changes the first frame **in place**.

> **What is a chunk?** A column is stored in memory as one or more contiguous blocks (chunks). One chunk is fastest to query. *Rechunking* copies the pieces into a single chunk. `df.n_chunks()` tells you how many there are.

`pl.concat(items, *, how="vertical", rechunk=True, parallel=True)` accepts a list of DataFrames, LazyFrames, or Series. The `how` argument chooses the strategy: `vertical`, `vertical_relaxed`, `horizontal`, `diagonal`, `diagonal_relaxed`, or `align`.

### Vertical

Stacks rows on top of each other. The frames must have the **same column names and data types**.

In [ ]:
rows_first = pl.DataFrame({"id": [1, 2, 3], "value": ["a", "b", "c"]})
rows_second = pl.DataFrame({"id": [4, 5], "value": ["d", "e"]})

pl.concat([rows_first, rows_second], how="vertical")

A common source of errors is when the frames have the same columns in a **different order**, or with **different names**. Vertical concatenation is strict about both:

In [ ]:
different_order = pl.DataFrame({"value": ["d", "e"], "id": [4, 5]})
different_names = pl.DataFrame({"id": [4, 5], "val": ["d", "e"]})


In [ ]:
pl.concat([rows_first, different_order], how="vertical")


In [ ]:
pl.concat([rows_first, different_names], how="vertical")

### Horizontal

Places frames side by side. Column names must be unique across the frames.

**This is where Polars 2.x differs from the book.** The book says that if the frames have different heights, the shorter one is padded with nulls. In Polars 2.1 the same call raised an error that mentions **strict** mode (this was visible in the saved output of the draft notebook, which was run on 2.1). First, frames of equal height, which always work:

In [ ]:
left_columns = pl.DataFrame({"id": [1, 2, 3], "value": ["a", "b", "c"]})
right_columns_equal = pl.DataFrame({"value2": ["x", "y", "z"]})

pl.concat([left_columns, right_columns_equal], how="horizontal")

Now frames of **different** heights (3 rows and 2 rows). The first cell shows what your version does by default; the second checks whether `concat` accepts a `strict` argument in your version. Read both outputs rather than assuming.

In [ ]:
right_columns_short = pl.DataFrame({"value2": ["x", "y"]})

In [ ]:
print('Default how="horizontal":')
pl.concat([left_columns, right_columns_short], how="horizontal")


In [ ]:
print("With strict=False (only if your version supports it):")
pl.concat([left_columns, right_columns_short], how="horizontal", strict=False)

In [ ]:
pl.concat([left_columns, right_columns_short], how="horizontal_extend")

### A robust way to pad: join on the row number

Whatever your version does by default, you can always get the padded result of the book **explicitly** by joining on a row index with a left join. This makes the intent clear (rows are matched by position, and the missing ones become null) and works the same in every version.

In [ ]:
padded = (
    left_columns.with_row_index("row")
    .join(right_columns_short.with_row_index("row"), on="row", how="left", validate="1:1")
    .sort("row")
    .drop("row")
)
padded

### Diagonal

The *union* of the columns: every column of every frame appears in the result, and cells that a frame doesn't have are filled with null. Columns are matched **by name**.

In [ ]:
diagonal_first = pl.DataFrame({"id": [1, 2, 3], "value": ["a", "b", "c"]})
diagonal_second = pl.DataFrame({"value": ["d", "e"], "value2": ["x", "y"]})

pl.concat([diagonal_first, diagonal_second], how="diagonal")

### Align

`how="align"` does not tape rows or columns together. It **matches rows by the values of the columns the frames have in common** (here `value`) and lines them up, like a full join that returns the combined table. Values with no partner get nulls.

In [ ]:
align_first = pl.DataFrame({"id": [1, 2, 3], "value": ["a", "b", "c"]})
align_second = pl.DataFrame({"value": ["a", "c", "d"], "value2": ["x", "y", "z"]})

pl.concat([align_first, align_second], how="align")

`how="align"` is built on `pl.align_frames()`, which keeps the frames **separate** but gives them identical rows for the chosen column (`on`), so you can compare them row by row. Its arguments: `*frames`, `on`, `how` (default `"full"`), `select`, `descending`.

Values missing in one frame become null there, and a value that appears **more than once** produces the **Cartesian product** of the matching rows. The key `2` appears twice in each frame below, so you get four `2` rows in each output.

In [ ]:
align_left = pl.DataFrame({"id": [1, 2, 2], "value": ["a", "c", "b"]})
align_right = pl.DataFrame({"id": [2, 2], "value": ["x", "y"]})

aligned_left, aligned_right = pl.align_frames(align_left, align_right, on="id")
print(aligned_left)
print(aligned_right)

### Relaxed: coercing to a common type

The `vertical`, `horizontal`, and `diagonal` strategies each have a `_relaxed` variant. Where columns with the same name have **different types**, a relaxed concat converts them to their common **supertype** (Int + Float becomes Float; Int + String becomes String). A strict concat raises a `SchemaError`.

In [ ]:
strict_first = pl.DataFrame({"id": [1, 2, 3], "value": ["a", "b", "c"]})
strict_second = pl.DataFrame({"id": [4.0, 5.0], "value": [1, 2]})


In [ ]:
print("how='vertical':")
pl.concat([strict_first, strict_second], how="vertical")

In [ ]:
relaxed = pl.concat([strict_first, strict_second], how="vertical_relaxed")

In [ ]:
print(relaxed.schema)

In [ ]:
relaxed

> **Use `_relaxed` with care.** Notice that the integers `1, 2` in `value` were silently turned into the *strings* `"1"`, `"2"`. Strict mode would have told you about the mismatch; relaxed mode papers over it. Prefer fixing the types **explicitly** (`cast`) so you decide the result type, and keep strict mode as a safety net.

## 7. Stacking: `vstack`, `hstack`, `append`, `extend`

`vstack` and `hstack` use the **"point, don't copy"** strategy. They are quick because no data is moved, but the result consists of **multiple chunks**, which can slow down later queries. (`pl.concat(..., how="vertical")` uses `vstack` internally and then rechunks.) These are the preferred tools when you append **many** frames one after another, and they only work on DataFrames, not LazyFrames.

`df.vstack(other)` requires the **same width, column names, and types**. It returns a **new** DataFrame and leaves the originals alone.

In [ ]:
stack_first = pl.DataFrame({"id": [1, 2], "value": ["a", "b"]})
stack_second = pl.DataFrame({"id": [3, 4], "value": ["c", "d"]})

stacked = stack_first.vstack(stack_second)

In [ ]:
stacked

### Chunks: the price of not copying

Compare the number of chunks after `vstack` (no copy) with `pl.concat` (copies and rechunks), and what `rechunk()` does.

In [ ]:
via_concat = pl.concat([stack_first, stack_second])  # rechunk=True by default

In [ ]:
print("chunks after vstack:        ", stacked.n_chunks())
print("chunks after pl.concat:     ", via_concat.n_chunks())
print("chunks after vstack+rechunk:", stacked.rechunk().n_chunks())


In [ ]:
assert_frame_equal(stacked, via_concat)


### `hstack`

`df.hstack(other)` puts columns side by side. It requires the **heights to match**.

In [ ]:
hstack_first = pl.DataFrame({"id": [1, 2], "value": ["a", "b"]})
hstack_second = pl.DataFrame({"value2": ["x", "y"]})

hstack_first.hstack(hstack_second)

In [ ]:
print("hstack with mismatched heights:")
hstack_first.hstack(pl.DataFrame({"value2": ["x", "y", "z"]}))

### Appending to a Series: `series.append()`

`Series.append(other)` adds the values of another Series. Like `vstack`, it does not copy the data into one chunk. **Careful: it modifies the Series in place.** The result keeps the **name of the first** Series. If you would rather not mutate, use `pl.concat([a, b])`.

In [ ]:
series_a = pl.Series("a", [1, 2])
series_b = pl.Series("b", [3, 4])

appended = series_a.append(series_b)
appended

In [ ]:
non_mutating = pl.concat([pl.Series("a", [1, 2]), pl.Series("b", [3, 4])])
non_mutating

### `extend`

`df.extend(other)` copies the rows of `other` **behind** the data of `df`, **when there is room** in memory. That keeps the data contiguous and can avoid a copy of the large frame, so it works best when you add a **small** frame to a **large** one.

> **In place is out of place.** `extend` modifies `df` itself. It returns the frame as well, but only for convenience. This is unlike almost everything else in Polars, so it is easy to trip over.

In [ ]:
extend_target = pl.DataFrame({"id": [1, 2], "value": ["a", "b"]})
extend_source = pl.DataFrame({"id": [3, 4], "value": ["c", "d"]})


In [ ]:
extend_target

In [ ]:
extend_source

In [ ]:
extend_target.extend(extend_source)


In [ ]:
extend_target

### Which one should I use?

| You want to... | Use | Notes |
|---|---|---|
| combine tables by matching keys | `df.join()` | add `validate=`; consider `maintain_order=` |
| match on the nearest value | `df.join_asof()` | sort first; mind `by`, `tolerance`, `strategy` |
| stack a few frames, then query a lot | `pl.concat([...])` | copies and rechunks: queries are fast afterwards |
| append many frames in a loop, DataFrames only | `df.vstack()` | no copy; many chunks, so rechunk at the end |
| add a small frame to a big one, mutating it | `df.extend()` | in place |
| combine lazy queries | `pl.concat([lf1, lf2])` | works on LazyFrames; stack methods do not |
| frames with different columns | `pl.concat(..., how="diagonal")` | union of columns, nulls elsewhere |
| frames with the same columns but different dtypes | `cast` first, or `how="vertical_relaxed"` | prefer explicit casts |

If you are tempted to build a DataFrame row by row in a Python loop, **collect the rows in a list** and make a single DataFrame (or one `pl.concat`) at the end. Repeated stacking, or worse repeated `concat`, is much slower.

## Takeaways

- `df.join()` combines frames by key. Choose the strategy deliberately (`inner`, `left`, `right`, `full`, `cross`, `semi`, `anti`); use `semi`/`anti` to **filter** by membership, and `coalesce=True` on a `full` join to merge the key columns.
- **Always validate** the relationship with `validate="m:1"`, `"1:m"`, or `"1:1"`: it turns silent row duplication into a clear error. The default `m:m` checks nothing.
- Row order of join results is **not guaranteed**: use `maintain_order=` or sort when order matters. Null keys do not match each other unless you opt in.
- Keys must have the **same dtype**; cast explicitly.
- `df.join_asof()` joins on the **nearest** value. Both frames must be **sorted** on the key; control it with `strategy` (`backward`, `forward`, `nearest`), `tolerance`, and `by` (exact-match groups).
- After any join, **sanity-check the row count** and ask why unmatched rows are unmatched (the Books campaign).
- `pl.concat()` combines with `how=` `vertical`, `horizontal`, `diagonal`, `align` (and `_relaxed` variants). It copies and rechunks. In Polars 2.x, horizontal concatenation of frames with different heights is **not silently padded**.
- `vstack`/`hstack`/`Series.append` avoid copying but leave multiple chunks; `extend` and `Series.append` change their target **in place**.

## Next up

Chapter 15 shows how to **reshape** DataFrames (wide to long and back) when you need to change the structure of your data.

## Try it yourself


1. **Anti join.** Which employees in the `employees` table work in a department that does **not** appear in `departments`? Which `departments` have **no** employees? Use two `anti` joins.


2. **Validation.** Join `employees` to `departments` with `validate="m:1"`, then remove the `department_id == 30` row from `departments` and repeat with `how="inner"` and `how="left"`. How do the row counts differ, and what shows up in `department_name` for Charlie?


3. **Full join.** Redo `df_left.join(df_right, on="key", how="full")` with and without `coalesce=True`. Which rows have nulls in which columns?


4. **Asof strategies.** For the `quotes` and `trades` tables, change the strategy to `"forward"` and `"nearest"`. What does each trade match now? What happens if you widen `tolerance` to `"10s"`?


5. **Campaigns.** Change the campaign window from 60 days to 30 and to 90 days. How do the average quantities change per campaign? Would you trust the Books conclusion with either window?


6. **Chunks.** Build a DataFrame by `vstack`-ing 100 one-row frames in a loop. Check `n_chunks()`, then call `rechunk()`. Then build the same thing with a single `pl.concat` of a list of 100 frames and compare timings with `%%timeit`.